# 06 - Physics-Informed Loss

**Architecture block 5.** `Loss = CE + lambda * Physics Loss`.

The physics loss never sees the forward epidemiological model - otherwise the
study would be circular. It states only weak directional facts.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent / "src"))

import cropforecast
from cropforecast.config import load_config, ensure_dirs, set_seed, Device
cfg = load_config(Path.cwd().parent / "configs" / "default.yaml")
ensure_dirs(cfg); set_seed(cfg.project.seed)
device = Device.auto(cfg.training.amp)
print("device:", device)

In [ ]:
import torch
from cropforecast.physics.losses import (
    PhysicsWeights, physics_loss, monotonic_horizon_loss,
    diffusion_loss, dry_suppression_loss, PHYSICS_COLUMNS)
print("physics inputs:", PHYSICS_COLUMNS)
print("weights       :", PhysicsWeights())

### Constraint 1 - forecast uncertainty must not shrink with horizon

In [ ]:
inc = torch.arange(4).float().repeat(16,1)     # uncertainty grows: legal
dec = -inc                                     # uncertainty shrinks: illegal
print("increasing uncertainty ->", monotonic_horizon_loss(inc).item(), "(no penalty)")
print("decreasing uncertainty ->", monotonic_horizon_loss(dec).item(), "(penalised)")

### Constraint 2 - risk is spatially smooth across connected farms

In [ ]:
ei = torch.randint(0, 64, (2, 300)); ew = torch.rand(300)
uniform = torch.ones(64, 4)
noisy   = torch.rand(64, 4)
print("uniform risk across farms ->", round(diffusion_loss(uniform, ei, ew).item(), 5))
print("random risk across farms  ->", round(diffusion_loss(noisy,  ei, ew).item(), 5))

### Constraint 3 - dry air suppresses fungal risk

In [ ]:
risk_high = torch.full((32,4), 0.9)
vpd_dry   = torch.full((32,), 2.0)    # strongly drying
vpd_humid = torch.full((32,), 0.3)
print("high risk + dry air   ->", round(dry_suppression_loss(risk_high, vpd_dry).item(), 4))
print("high risk + humid air ->", round(dry_suppression_loss(risk_high, vpd_humid).item(), 4))

### All terms together

In [ ]:
torch.manual_seed(0)
risk = torch.rand(64,4, requires_grad=True); logvar = torch.randn(64,4, requires_grad=True)
phys = torch.stack([torch.rand(64)*20, 40+torch.rand(64)*55, torch.rand(64)*2.5, 10+torch.rand(64)*25],1)
total, parts = physics_loss(risk, logvar, phys, ei, ew)
print("total:", round(total.item(),4)); parts

### Training curve

How each physics term evolves. Run stage 4 first.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
hist_path = Path(cfg.paths.reports) / "stage4_history.csv"
if hist_path.exists():
    h = pd.read_csv(hist_path)
    cols = [c for c in h.columns if c.startswith("phys_")]
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    for c in cols: ax[0].plot(h.epoch, h[c], label=c.replace("phys_",""))
    ax[0].set_title("physics constraint violations"); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(h.epoch, h.val_macro_f1, label="val macro-F1")
    ax[1].plot(h.epoch, h.val_risk_r2_mean, label="val risk R2")
    ax[1].set_title("validation"); ax[1].legend(); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()
else:
    print("Run scripts/04_train_and_ablate.py")